# Alternative Sentiment Scores

Calculate FinBERT sentiment probabilities for the filtered AAPL-only, high-information news rows. Reuse a cached artifact only when its ordered news IDs exactly match the input.

## Process the Data

- FinBERT scores the ordered headline-plus-summary text with a batch size of 16, a throughput and memory compromise that does not change the model itself.
- Cached scores are reused only when their ordered news IDs exactly match the input, preventing a stale row alignment while avoiding repeated inference.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.data_preprocessing.alternative_sentiment_scores import (
    score_sentiment_features,
)

PROJECT_ROOT = Path.cwd().resolve().parents[1]
data_root = PROJECT_ROOT / "data/research_data/alternative"
period = "2025-01-01_2025-12-31"
news_path = data_root / "data" / f"aapl_{period}.parquet"
sentiment_path = data_root / "features" / f"aapl_finbert_sentiment_scores_{period}.parquet"
alternative_data = pd.read_parquet(news_path)
score_columns = [
    "sentiment_positive", "sentiment_negative",
    "sentiment_neutral", "sentiment_score",
]
cache_matches_input = False
if sentiment_path.is_file():
    sentiment_features = pd.read_parquet(sentiment_path)
    cache_matches_input = (
        "id" in sentiment_features
        and sentiment_features["id"].tolist() == alternative_data["id"].tolist()
    )
if not cache_matches_input:
    sentiment_features = score_sentiment_features(
        alternative_data,
        batch_size=16,
    )
    sentiment_path.parent.mkdir(parents=True, exist_ok=True)
    sentiment_features.to_parquet(sentiment_path, index=False)
sentiment_path

PosixPath('/Users/kwonjunhyuk9/Documents/financial-machine-learning/data/research_data/alternative/features/aapl_finbert_sentiment_scores_2025-01-01_2025-12-31.parquet')

## Take a Quick Look at the Data Structure

- These read-only checks verify the retained schema, source coverage, and positive, negative, neutral, and signed sentiment-score distributions.
- Histogram bins and figure dimensions change only the display, not the stored probabilities.

In [2]:
sentiment_features.head()

,id,headline,source,url,summary,created_at,updated_at,symbols,author,content,sentiment_positive,sentiment_negative,sentiment_neutral,sentiment_score
0,42767369,'CPCS Secures Agreement With Apple For Enhance...,benzinga,https://www.benzinga.com/news/25/01/42767369/c...,,2025-01-02 15:32:15+00:00,2025-01-02 15:32:15+00:00,AAPL,Benzinga Newsdesk,,0.928484,0.013745,0.057771,0.914739
1,42784735,"B of A Securities Maintains Buy on Apple, Main...",benzinga,https://www.benzinga.com/news/25/01/42784735/b...,,2025-01-03 13:57:22+00:00,2025-01-03 13:57:22+00:00,AAPL,Benzinga Newsdesk,,0.035534,0.019706,0.944760,0.015828
2,42788545,"Bernstein Maintains Outperform on Apple, Raise...",benzinga,https://www.benzinga.com/news/25/01/42788545/b...,,2025-01-03 15:30:38+00:00,2025-01-03 15:30:39+00:00,AAPL,Benzinga Newsdesk,,0.888683,0.046740,0.064577,0.841944
3,42840188,"MoffettNathanson Downgrades Apple to Sell, Low...",benzinga,https://www.benzinga.com/news/25/01/42840188/m...,,2025-01-07 12:00:17+00:00,2025-01-07 12:00:18+00:00,AAPL,Benzinga Newsdesk,,0.072308,0.451398,0.476293,-0.379090
4,42920303,"Analyst Ming-Chi Kuo Says ""Apple Likely To Fac...",benzinga,https://www.benzinga.com/news/25/01/42920303/a...,,2025-01-10 17:01:00+00:00,2025-01-10 17:01:01+00:00,AAPL,Benzinga Newsdesk,,0.882844,0.086813,0.030343,0.796031


In [3]:
sentiment_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 348 entries, 0 to 347
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   id                  348 non-null    int64              
 1   headline            348 non-null    str                
 2   source              348 non-null    str                
 3   url                 348 non-null    str                
 4   summary             348 non-null    str                
 5   created_at          348 non-null    datetime64[us, UTC]
 6   updated_at          348 non-null    datetime64[us, UTC]
 7   symbols             348 non-null    str                
 8   author              348 non-null    str                
 9   content             348 non-null    str                
 10  sentiment_positive  348 non-null    float64            
 11  sentiment_negative  348 non-null    float64            
 12  sentiment_neutral   348 non-null    float64    

In [4]:
sentiment_features["source"].value_counts(dropna=False)

source
benzinga    348
Name: count, dtype: int64

In [5]:
sentiment_features[score_columns].describe()

,sentiment_positive,sentiment_negative,sentiment_neutral,sentiment_score
count,348.000000,348.000000,348.000000,348.000000
mean,0.332261,0.215340,0.452400,0.116921
std,0.341588,0.318874,0.349479,0.560884
min,0.007474,0.006566,0.012556,-0.963803
25%,0.041293,0.016013,0.094637,-0.234545
50%,0.156505,0.033167,0.385231,0.088608
75%,0.625044,0.325927,0.836272,0.595988
max,0.957127,0.973652,0.949904,0.941092


In [6]:
sentiment_features[score_columns].hist(bins=30, figsize=(10, 8))
plt.tight_layout()
plt.show()

/var/folders/1z/bcvql7210c77v6rjkswpzsyr0000gn/T/ipykernel_82322/2085060402.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
